## Celda 14 — Extracción de Embeddings (Fase 3, paso 1/4)

Pegar al final del notebook. Reutiliza `transform_val`, `class_names` (Celda 8), `device` (Celda 9), `DATA_PROCESSED` y `PROJECT_DIR` (Celda 1), y el archivo `best_model_finetuned.pth`.

In [ ]:
# === CELDA 14: Extracción de Embeddings (Fase 3) ===
# Convierte cada imagen en su "huella visual": el vector de 1280 dimensiones que
# la red aprendió, ANTES del clasificador. Sobre estos vectores vamos a montar el
# detector de anomalías. Se guardan en Drive para no recalcularlos cada vez.
import os
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets
from torchvision.models import efficientnet_b0
from torch.utils.data import DataLoader

MODELO_FT_PATH = os.path.join(PROJECT_DIR, "best_model_finetuned.pth")

# 1) Reconstruir la arquitectura y cargar NUESTROS pesos fine-tuneados.
#    weights=None: no bajamos los de ImageNet, cargamos los del .pth.
extractor = efficientnet_b0(weights=None)
extractor.classifier[1] = nn.Linear(extractor.classifier[1].in_features, len(class_names))
extractor.load_state_dict(torch.load(MODELO_FT_PATH, map_location=device))

# 2) "Decapitar" la red: reemplazar el clasificador por Identity. Ahora el modelo
#    devuelve el embedding de 1280-D (la capa de pooling), no las 4 probabilidades.
extractor.classifier = nn.Identity()
extractor = extractor.to(device).eval()

# 3) Datasets DETERMINISTAS: mismo transform que validación (Letterbox + Normalize,
#    SIN augmentation). Para los embeddings queremos la representación estable de
#    cada imagen, no una versión rotada/jiteada al azar.
ds_train = datasets.ImageFolder(os.path.join(DATA_PROCESSED, "train"), transform=transform_val)
ds_val   = datasets.ImageFolder(os.path.join(DATA_PROCESSED, "val"),   transform=transform_val)


def extraer_embeddings(dataset, etiqueta):
    loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2)
    embs, labels = [], []
    with torch.no_grad():
        for imgs, y in loader:
            z = extractor(imgs.to(device))      # (B, 1280)
            embs.append(z.cpu().numpy())
            labels.append(y.numpy())
    embs = np.concatenate(embs)
    labels = np.concatenate(labels)
    print(f"   {etiqueta:5s}: embeddings {embs.shape} | etiquetas {labels.shape}")
    return embs, labels


print("Extrayendo embeddings de 1280-D...")
X_train, y_train = extraer_embeddings(ds_train, "train")
X_val,   y_val   = extraer_embeddings(ds_val,   "val")

# 4) Guardar en Drive (con rutas y nombres de clase, para poder rastrear después
#    QUÉ imagen marca el detector como anomalía).
rutas_train = [p for p, _ in ds_train.samples]
rutas_val   = [p for p, _ in ds_val.samples]
np.savez(os.path.join(PROJECT_DIR, "embeddings_train.npz"),
         X=X_train, y=y_train, rutas=rutas_train, clases=class_names)
np.savez(os.path.join(PROJECT_DIR, "embeddings_val.npz"),
         X=X_val, y=y_val, rutas=rutas_val, clases=class_names)
print(f"\nGuardados: embeddings_train.npz ({X_train.shape}) y embeddings_val.npz ({X_val.shape})")
print("class_names:", class_names)
